# 06. 장기 메모리 & 스킬

## 학습 목표
- `CompositeBackend` + `StoreBackend`로 장기 메모리를 구현한다
- 크로스 스레드 메모리 공유 패턴을 이해한다
- `AGENTS.md`를 통해 에이전트에 컨텍스트를 주입한다
- 스킬(SKILL.md)의 구조와 Progressive Disclosure를 이해한다
- Skills vs Memory의 차이를 파악한다

In [1]:
# 환경 설정
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY가 설정되지 않았습니다!"
print("환경 설정 완료")

환경 설정 완료


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault(
        "LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default")
    )
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler

    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


LangSmith tracing ON — project: LangGraph-Tutorial


In [3]:
# OpenAI gpt-4.1 모델 설정
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5.4-mini")

/home/sds/projects/SDS-AX-Advanced-2026-1/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 1. 장기 메모리가 필요한 이유

기본 `StateBackend` 에이전트는 **대화 스레드가 끝나면 모든 정보를 잊습니다**.  
하지만 실제 어시스턴트는 아래 정보를 **대화 간에 유지**해야 합니다:

- 사용자 선호도 (코딩 스타일, 사용 언어)
- 프로젝트 컨벤션 (아키텍처 결정, 네이밍 규칙)
- 이전 대화에서 학습한 피드백
- 자주 참조하는 정보 (API 문서, 설정값)

### 해결 방식: CompositeBackend

![CompositeBackend 라우팅](../assets/images/composite_backend.png)

`/memories/` 경로에 저장된 파일은 **어떤 대화 스레드에서든** 접근할 수 있습니다.

In [4]:
from deepagents import create_deep_agent
from deepagents.backends import (
    StateBackend,
    StoreBackend,
    CompositeBackend,
    FilesystemBackend,
)
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver


# 1. 스토어와 체크포인터 생성
store = InMemoryStore()  # 개발용 (프로덕션: PostgresStore)
checkpointer = MemorySaver()  # 에이전트 상태 유지


# 2. CompositeBackend 팩토리 — /memories/만 영속, 나머지는 에페메럴
def memory_backend_factory(runtime):
    return CompositeBackend(
        default=StateBackend(runtime),
        routes={
            "/memories/": StoreBackend(runtime),
        },
    )


# 3. 에이전트 생성
memory_agent = create_deep_agent(
    model=model,
    system_prompt="""당신은 개인 어시스턴트입니다.
사용자가 기억해 달라고 하는 정보는 /memories/ 폴더에 저장하세요.
이전에 저장한 메모리가 있으면 참고하여 응답하세요.
한국어로 응답하세요.""",
    backend=memory_backend_factory,
    store=store,
    checkpointer=checkpointer,
)

print("장기 메모리 에이전트 생성 완료")

장기 메모리 에이전트 생성 완료


---
## 2. 크로스 스레드 메모리 공유

`StoreBackend`에 저장된 데이터는 **스레드 간에 공유**됩니다.  
아래 예제에서 스레드 1에서 저장한 선호도를 스레드 2에서 읽어봅니다.

In [9]:
# 스레드 1: 사용자 선호도 저장
config_t1 = {"configurable": {"thread_id": "memory-thread-1"}}

result1 = memory_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
다음 정보를 기억해 주세요:
- 선호하는 프로그래밍 언어: Python
- 선호하는 에디터: VS Code
- 코딩 스타일: Black 포맷터, 타입 힌트 필수.
""",
            }
        ]
    },
    config={**config_t1, **lf_config},
)

print("[스레드 1] 저장 결과:")
print(result1["messages"][-1].content)

[스레드 1] 저장 결과:
이미 저장되어 있습니다. 같은 내용으로 `/memories/preferences.txt`에 반영되어 있습니다.


In [10]:
# InMemoryStore에 저장된 메모리 확인
print("=== InMemoryStore 저장 확인 ===\n")

# 1. 등록된 네임스페이스 목록 조회
namespaces = store.list_namespaces(prefix=())
print(f"등록된 네임스페이스: {namespaces}\n")

# 2. 각 네임스페이스별 저장된 항목 조회
for ns in namespaces:
    items = store.search(ns, limit=100)
    print(f"--- 네임스페이스: {ns} ({len(items)}개 항목) ---")
    for item in items:
        print(f"  키: {item.key}")
        print(f"  값: {item.value}")
        print(f"  생성일: {item.created_at}")
        print(f"  수정일: {item.updated_at}")
        print()

# 3. store 내부 raw 데이터 직접 확인 (디버깅용)
print("=== Store 내부 raw 데이터 ===")
for ns, items_dict in store._data.items():
    print(f"\n네임스페이스: {ns}")
    for key, val in items_dict.items():
        print(f"  키: {key}")
        print(f"  값: {val}")


=== InMemoryStore 저장 확인 ===

등록된 네임스페이스: [('filesystem',)]

--- 네임스페이스: ('filesystem',) (1개 항목) ---
  키: /preferences.txt
  값: {'content': ['선호하는 프로그래밍 언어: Python', '선호하는 에디터: VS Code', '코딩 스타일: Black 포맷터, 타입 힌트 필수', ''], 'created_at': '2026-04-01T04:13:33.939167+00:00', 'modified_at': '2026-04-01T04:13:33.939167+00:00'}
  생성일: 2026-04-01 04:13:33.939188+00:00
  수정일: 2026-04-01 04:13:33.939188+00:00

=== Store 내부 raw 데이터 ===

네임스페이스: ('filesystem',)
  키: /preferences.txt
  값: Item(namespace=['filesystem'], key='/preferences.txt', value={'content': ['선호하는 프로그래밍 언어: Python', '선호하는 에디터: VS Code', '코딩 스타일: Black 포맷터, 타입 힌트 필수', ''], 'created_at': '2026-04-01T04:13:33.939167+00:00', 'modified_at': '2026-04-01T04:13:33.939167+00:00'}, created_at='2026-04-01T04:13:33.939188+00:00', updated_at='2026-04-01T04:13:33.939188+00:00')


In [11]:
# 스레드 2: 다른 대화에서 저장된 메모리 활용
config_t2 = {"configurable": {"thread_id": "memory-thread-2"}}

result2 = memory_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "/memories/preferences.txt를 읽어서 내 선호도를 알려 주세요.",
            }
        ]
    },
    config={**config_t2, **lf_config},
)

print("[스레드 2] 메모리 읽기 결과:")
print(result2["messages"][-1].content)

[스레드 2] 메모리 읽기 결과:
기억된 선호도는 다음과 같습니다.

- 선호하는 프로그래밍 언어: Python
- 선호하는 에디터: VS Code
- 코딩 스타일: Black 포맷터, 타입 힌트 필수

원하시면 이 선호도를 바탕으로 앞으로 답변이나 코드 작성 스타일도 맞추겠습니다.


---
## 3. AGENTS.md를 통한 컨텍스트 주입

`memory` 파라미터를 사용하면, 에이전트가 시작될 때 **AGENTS.md 파일을 자동으로 로드**하여  
시스템 프롬프트에 주입합니다.

### AGENTS.md란?
에이전트에게 항상 적용되어야 하는 **규칙, 컨벤션, 컨텍스트 정보**를 담는 마크다운 파일입니다.

### 특징
- 에이전트가 시작할 때 **항상 로드** (on-demand가 아님)
- `<agent_memory>` 태그로 시스템 프롬프트에 주입
- 여러 소스 지정 가능 (배열)
- 에이전트가 `edit_file` 도구로 AGENTS.md를 **스스로 업데이트** 가능

In [18]:
import tempfile
from pathlib import Path

# 임시 디렉토리 생성 — FilesystemBackend의 root_dir로 사용
tmp_dir = tempfile.mkdtemp()
print(f"임시 디렉토리 생성: {tmp_dir}")

# 1) AGENTS.md 파일 생성 — memory 파라미터로 시스템 프롬프트에 주입됨
agents_md_path = Path(tmp_dir) / "memory" / "AGENTS.md"
agents_md_path.parent.mkdir(parents=True, exist_ok=True)
agents_md_path.write_text("""\
# 프로젝트 컨벤션

## 코딩 규칙
- 모든 함수에 타입 힌트를 작성할 것
- docstring은 Google 스타일을 사용할 것
- 변수명은 snake_case를 사용할 것

## 아키텍처
- 레이어드 아키텍처 (Controller → Service → Repository)
- 의존성 주입(DI) 패턴 사용

## 보안
- SQL 쿼리는 반드시 파라미터 바인딩 사용
- 사용자 입력은 항상 검증할 것
""", encoding="utf-8")

# 2) 최소 프로젝트 구조 생성 — 에이전트가 탐색할 수 있도록
src_dir = Path(tmp_dir) / "src"
src_dir.mkdir(parents=True, exist_ok=True)
(src_dir / "__init__.py").write_text("", encoding="utf-8")

print(f"AGENTS.md 생성: {agents_md_path}")
print(f"프로젝트 구조 생성: {src_dir}")

# 생성된 디렉토리 구조 확인
for p in sorted(Path(tmp_dir).rglob("*")):
    rel = p.relative_to(tmp_dir)
    prefix = "📁" if p.is_dir() else "📄"
    print(f"  {prefix} /{rel}")

임시 디렉토리 생성: /tmp/tmpse0np1u9
AGENTS.md 생성: /tmp/tmpse0np1u9/memory/AGENTS.md
프로젝트 구조 생성: /tmp/tmpse0np1u9/src
  📁 /memory
  📄 /memory/AGENTS.md
  📁 /src
  📄 /src/__init__.py


In [19]:
# memory 파라미터로 AGENTS.md 로드
memory_inject_agent = create_deep_agent(
    model=model,
    system_prompt="당신은 코딩 어시스턴트입니다.",
    backend=FilesystemBackend(root_dir=tmp_dir, virtual_mode=True),
    memory=["/memory/AGENTS.md"],  # AGENTS.md 파일 경로
)

# 에이전트가 AGENTS.md의 규칙을 따르는지 확인
result = memory_inject_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "간단한 HTTP 클라이언트 클래스를 만들어 주세요. 프로젝트 컨벤션을 따라주세요.",
            }
        ]
    },
    config=lf_config,
)

print(result["messages"][-1].content)

`/src/http_client.py`에 간단한 `HTTPClient` 클래스를 추가했습니다.

포함 내용:
- `get()`
- `post()`
- 내부 `_request()`
- 내부 `_build_url()`

컨벤션 반영:
- 타입 힌트 추가
- Google 스타일 docstring
- `snake_case` 사용

주의:
- 현재 `post()`는 JSON 형태의 간단한 구현으로 넣었지만, 본문 직렬화는 매우 단순합니다.
- 필요하면 다음 단계로 `json` 모듈을 써서 진짜 JSON 바디로 보내도록 개선할 수 있습니다.


---
## 4. 스킬 (Skills)

스킬은 에이전트에게 **도메인 전문 지식**을 부여하는 모듈화된 지침 세트입니다.

### Memory vs Skills

| 비교 | Memory (AGENTS.md) | Skills (SKILL.md) |
|------|-------------------|-------------------|
| **로딩** | 항상 로드 (Always) | 필요 시 로드 (On-demand) |
| **파일 형식** | `AGENTS.md` | `SKILL.md` (YAML 프론트매터) |
| **적합한 용도** | 항상 적용되는 규칙/컨벤션 | 특정 태스크에 필요한 큰 컨텍스트 |
| **토큰 효율** | 항상 소비 | 점진적 공개로 절약 |
| **크기** | 간결할수록 좋음 | 대용량 가능 (10MB 제한) |
| **업데이트** | 에이전트가 edit_file로 수정 가능 | 보통 정적 |

### Progressive Disclosure (점진적 공개)

스킬은 한 번에 전부 로드하지 않습니다:
1. 처음에는 **프론트매터(이름, 설명)**만 로드
2. 사용자 요청과 관련된 스킬을 **에이전트가 판단**
3. 필요한 스킬의 **전체 내용**을 그때 로드

이 방식으로 토큰을 절약하면서도 필요한 전문 지식에 접근할 수 있습니다.

### SKILL.md 구조

```yaml
---
name: web-research           # 스킬 식별자 (최대 64자, 소문자+하이픈)
description: >               # 설명 (최대 1024자) — 매칭에 사용
  체계적인 웹 리서치를 수행하기 위한 단계별 가이드.
  정보 수집, 검증, 정리까지의 전체 워크플로를 다룹니다.
license: MIT
compatibility: Python 3.8+
metadata:
  category: research
allowed-tools: ls read_file write_file
---

# Web Research 스킬

## 사용 시기
- 사용자가 특정 주제에 대한 조사를 요청할 때
- 최신 정보가 필요한 질문이 들어올 때

## 워크플로
1. 검색 쿼리 설계
2. 다양한 소스에서 정보 수집
3. 정보 교차 검증
4. 구조화된 보고서 작성
```

In [22]:
# 스킬을 사용하는 에이전트 생성
# Day-03/skills/ 폴더의 실제 SKILL.md를 참조하도록 설정
from pathlib import Path

skills_root = str(Path.cwd().parent)  # deepagents_study의 parent = Day-03
print(f"스킬 루트 디렉토리: {skills_root}")
print(f"code-review 스킬 경로: {Path(skills_root) / 'skills' / 'code-review' / 'SKILL.md'}")
print(f"스킬 파일 존재 여부: {(Path(skills_root) / 'skills' / 'code-review' / 'SKILL.md').exists()}")

skilled_agent = create_deep_agent(
    model=model,
    system_prompt="당신은 시니어 개발자입니다. 사용 가능한 스킬을 활용하여 작업을 수행하세요. 한국어로 응답하세요.",
    backend=FilesystemBackend(root_dir=skills_root, virtual_mode=True),
    skills=["/skills/"],  # Day-03/skills/ 폴더를 참조
)

print("\n스킬 에이전트 생성 완료")

스킬 루트 디렉토리: /home/sds/projects/SDS-AX-Advanced-2026-1/Day-03
code-review 스킬 경로: /home/sds/projects/SDS-AX-Advanced-2026-1/Day-03/skills/code-review/SKILL.md
스킬 파일 존재 여부: True

스킬 에이전트 생성 완료


In [23]:
# code-review 스킬을 활용한 코드 리뷰 요청
result = skilled_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
code-review 스킬을 사용하여 다음 코드를 리뷰해 주세요.
7가지 분석 관점(정확성, 유지보수성, 가독성, 효율성, 보안, 엣지 케이스, 테스트 커버리지)에서
피드백 구조(요약, 발견사항, 권고사항)에 맞춰 리뷰해 주세요.

```python
import os

def process_data(data):
    result = []
    for item in data:
        if item['type'] == 'user':
            name = item['name']
            email = item['email']
            query = f"INSERT INTO users VALUES ('{name}', '{email}')"
            db.execute(query)
            result.append(item)
    return result
```
""",
            }
        ]
    },
    config=lf_config,
)

print(result["messages"][-1].content)

Ignoring non-string 'allowed-tools' in /skills/account-recovery/SKILL.md (got list)
Ignoring non-string 'allowed-tools' in /skills/kb-search/SKILL.md (got list)


## 요약
이 코드는 현재 상태로는 안전하지 않고, 실행 시 오류 가능성이 높습니다. 특히 `db`가 정의되어 있지 않고, 문자열 보간으로 SQL을 직접 구성해 SQL Injection 위험이 큽니다. 또한 입력 검증과 예외 처리가 없어 데이터 품질이 조금만 나빠져도 쉽게 깨집니다.

## 발견사항

### Critical
1. **정의되지 않은 `db` 사용**
   - `db.execute(query)`에서 `db`가 함수 인자나 로컬/전역으로 정의되어 있지 않습니다.
   - 이 코드는 현재 형태로는 `NameError`가 발생합니다.
   - **관점**: 정확성, 테스트 가능성

2. **SQL Injection 취약점**
   - `name`, `email`을 f-string으로 직접 SQL에 삽입하고 있습니다.
   - 입력값에 작은따옴표나 악의적 SQL이 들어가면 쿼리 조작이 가능합니다.
   - **관점**: 보안, 정확성

3. **키 누락 시 즉시 예외 발생**
   - `item['type']`, `item['name']`, `item['email']`은 키가 없으면 `KeyError`를 발생시킵니다.
   - 외부 입력 데이터라면 매우 흔한 실패 지점입니다.
   - **관점**: 엣지 케이스, 정확성

### Improvements
4. **책임 분리가 되어 있지 않음**
   - `process_data`가 데이터 필터링, SQL 생성, DB 저장, 결과 수집을 모두 담당합니다.
   - 유지보수와 테스트가 어려워집니다.
   - **관점**: 유지보수성, 테스트 가능성

5. **불필요한 `os` import**
   - `os`는 사용되지 않습니다.
   - **관점**: 가독성, 유지보수성

6. **반복문 내 DB 호출로 인한 성능 저하 가능성**
   - 사용자 수가 많으면 항목마다 개별 insert가 발생합니다.
   - 배치 insert 또는 트랜잭션 처리를 고려하는 것이 좋습니다.
   - **관점**: 효율성

7. 

---
## 5. 스킬 소스 우선순위

여러 스킬 소스를 지정하면, **나중에 나온 소스가 우선**합니다 (last wins).

```python
skills=[
    "/skills/base/",     # 기본 스킬
    "/skills/user/",     # 사용자 스킬 (base 덮어쓰기 가능)
    "/skills/project/",  # 프로젝트 스킬 (최우선)
]
```

같은 이름의 스킬이 여러 소스에 있으면, 마지막 소스의 스킬이 사용됩니다.

---
## 6. 서브에이전트의 스킬 상속

| 서브에이전트 유형 | 스킬 상속 |
|-------------------|----------|
| General-purpose (빌트인) | 메인 에이전트의 스킬을 **자동 상속** |
| 커스텀 SubAgent | **명시적 `skills` 파라미터** 필요 |

```python
# 커스텀 서브에이전트에 스킬 부여
subagent = {
    "name": "reviewer",
    "description": "코드 리뷰 전문 에이전트",
    "system_prompt": "...",
    "tools": [],
    "skills": ["/skills/code-review/"],  # 명시적 스킬 경로
}
```

---
## 핵심 정리

| 항목 | 내용 |
|------|------|
| 장기 메모리 | `CompositeBackend` + `StoreBackend`로 `/memories/` 영속화 |
| AGENTS.md | `memory=["/path/AGENTS.md"]` → 항상 시스템 프롬프트에 주입 |
| Skills | `skills=["/skills/"]` → SKILL.md 기반 점진적 공개 |
| Progressive Disclosure | 프론트매터만 먼저 로드 → 필요 시 전체 로드 |
| 스킬 우선순위 | 나중 소스가 우선 (last wins) |
| Memory vs Skills | Memory = 항상 로드 / Skills = 필요 시 로드 |

## 다음 단계
→ **[07_advanced.ipynb](./07_advanced.ipynb)**: Human-in-the-Loop, 스트리밍, 샌드박스 등 고급 기능을 배웁니다.